# Data preparation and source exploration

County Health Rankings 2025 and HUD CHAS Table 7. Run top to bottom from the repository root or src/ directory. The CHAS data are explored descriptively, not merged into the model dataset.

In [ ]:
from pathlib import Path

# Run from the repository root or its src/ folder.
# If using Colab, upload/open the repository files in a runtime where
# the data/ directory is available; no personal Google Drive path is required.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "NM-Mental-Health-Housing-Stability"]
project_dir = next((p.resolve() for p in candidates if (p / "data" / "raw").is_dir()), None)
if project_dir is None:
    raise FileNotFoundError(
        "Could not find data/raw. Run from the repository root or src/ folder "
        "with the project's data/ directory available."
    )
%cd {project_dir}
print("Project:", project_dir)


## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

## The datasets
- '2025 CHR CSV Analytic Data'
  - https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation

- HUD CHAS

In [ ]:
file_path = 'data/raw/2025_county_health_analytic_data2025_v3.csv'

In [ ]:
df = pd.read_csv(file_path)
df.shape

In [ ]:
df.head()

In [ ]:
with open(file_path) as f:
  for i in range(10):
    print(f.readline())

In [ ]:
# read in with the first line as header
df = pd.read_csv(file_path, header=0)

In [ ]:
print(df.shape)
print(df.columns[:20])

## Data Cleaning

In [ ]:
# get the nm county data
nm = df[df['State Abbreviation'] == 'NM'].copy()

In [ ]:
print(nm.shape)
nm[['Name']].head()

## Preliminary Analysis of NM County Health Rankings
- Higher values of Mental Health Providers raw value means better access
- The measure appears to be providers per population rather than provider-to-population ratio
- Poor Mental Health Days range is about 5-6.5 days

## CHAS Table 7 Data

In [ ]:
!pwd

In [ ]:
!ls -a

In [ ]:
# get the CHAS dataset
file_path2 = 'data/raw/CHAS_Table7.csv'

In [ ]:
df2 = pd.read_csv(file_path2)

In [ ]:
df2.head()

In [ ]:
df2.shape

In [ ]:
# filter for the 33 new mexico counties
nm_chas = df2[df2['st'] == 35].copy()

In [ ]:
nm_chas.head()

In [ ]:
print(nm_chas.shape)
print(nm_chas['name'].head())

### According to the CHAS-data-dictionary-18-22
| Variable    | Meaning                       |
| ----------- | ----------------------------- |
| `T7_est1`   | Total occupied housing units  |
| `T7_est2`   | Owner-occupied housing units  |
| `T7_est108` | Renter-occupied housing units |


In [ ]:
# get the total occupied housing units, owner-occupied units, and renter-occupied units
nm_chas[['name', 'T7_est1', 'T7_est2', 'T7_est108']].head()

We will only use the county health dataset

## EDA

In [ ]:
[col for col in nm.columns if 'Housing' in col]

In [ ]:
# get the features we need
nm_subset = nm[[
    'Name',
    'Severe Housing Cost Burden raw value',
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Children in Poverty raw value',
    'Unemployment raw value',
    'Income Inequality raw value',
    'Child Care Cost Burden raw value',
    'Median Household Income raw value'
]]

nm_subset.head()

In [ ]:
# get summary stats
nm_subset.describe().transpose()

In [ ]:
# check datatypes
nm_subset.dtypes

In [ ]:
# convert datatypes to numeric
numeric_cols = nm_subset.columns.drop('Name')

nm_subset[numeric_cols] = nm_subset[numeric_cols].apply(
    pd.to_numeric,
    errors='coerce',
    )

In [ ]:
# check that datatypes were converted
nm_subset.dtypes

In [ ]:
nm_subset = nm_subset[
    nm_subset['Name'] != 'New Mexico'
].copy()

nm_subset.shape

In [ ]:
# check for nulls
nm_subset.isna().sum()

In [ ]:
# get summary stats with the converted datatypes
nm_subset.describe().transpose()

## Export clean county data

This file is the input to the analysis and modeling notebook.

In [ ]:
from pathlib import Path
Path('data/processed').mkdir(parents=True, exist_ok=True)
nm_subset.to_csv('data/processed/nm_county_clean.csv', index=False)
print('Saved', nm_subset.shape, 'to data/processed/nm_county_clean.csv')